In [2]:
import sys
import os

path_to_scripts = os.path.join('..', '..', 'python_scripts')
sys.path.append(path_to_scripts)

%load_ext autoreload

In [49]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_error, max_error

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from data_to_bigquery import load_from_bigquery
from feature_engineering import drop_lag_nulls, validate_features, engineer_features
from baseline_model import baseline_model_xgb, xgb_train_preproc, evaluate_trained_model, pred_preproc_xgb, xgb_prediction


%matplotlib inline

# Function checking 

In [ ]:
# raw table
df_raw = load_from_bigquery('gridzero-489711', 'merged_set', 'test_merge_2017_onward_raw')
# display(df_raw.head())
# df_raw.columns

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 148,991 rows and 26 columns from gridzero-489711.merged_set.test_merge_2017_onward_raw


In [ ]:
# train preproc function
df_pre = xgb_train_preproc(df_raw, add_year_lag=True)
# success working!

In [ ]:
# checking train_preproc
print(df_raw.shape)
print(df_pre.shape)
print(df_pre.columns.tolist())
print(df_pre.isna().sum().sum())
# success working!

(148991, 26)
(126157, 33)
['datetime', 'temperature_2m_c', 'wind_speed_100m_ms', 'wind_gusts_10m_ms', 'cloud_cover_pct', 'shortwave_radiation_wm2', 'direct_radiation_wm2', 'diffuse_radiation_wm2', 'pressure_msl_hpa', 'snowfall_cm', 'rain_mm', 'precipitation_mm', 'Biomass', 'Fossil Gas', 'Fossil Hard coal', 'Fossil Oil', 'Hydro Pumped Storage', 'Hydro Run-of-river and poundage', 'Nuclear', 'Other', 'Solar', 'Wind Offshore', 'Wind Onshore', 'TotalOutput-MW', 'carbon_intensity_gCO2_kWh', 'status', 'lag_48', 'lag_336', 'lag_17520', 'hour', 'day_of_week', 'month', 'is_weekend']
0


In [ ]:
# checking model function
model, X_train, X_test, y_train, y_test = baseline_model_xgb()
# success working!

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 148,991 rows and 26 columns from gridzero-489711.merged_set.test_merge_2017_onward_raw


In [ ]:
# evaluate trained model
metrics = evaluate_trained_model(model, X_test, y_test)
print(metrics)

{'MAE': 8.156732706083464, 'RMSE': 10.66453714130536, 'R2': 0.9802316395479151, 'MaxError': 102.64424133300781}


In [ ]:
# pred_preproc using X_test as df_new to check fucntion is working
#takes model ouput X_train
feature_cols = X_train.columns.tolist()

X_new = pred_preproc_xgb(df_new=X_test.copy(),
                     feature_cols=feature_cols
                     )

print(X_new.shape)
print(X_new.columns.tolist() == feature_cols)
print(X_new.isna().sum().sum())

(43035, 29)
True
0


In [ ]:
# checking prediction fucntion
prediction = xgb_prediction(
            model=model,
            df_new=X_test.copy(),
            feature_cols=feature_cols
            )

print(type(prediction))
print(len(prediction))
print(prediction[:5])

<class 'numpy.ndarray'>
43035
[229.6808  146.74657 212.47081 159.01811 309.9817 ]


## CVGridsearch for params

In [ ]:
# BASELINE
#
# pipeline = pipeline = make_pipeline(
#         StandardScaler(),
#         XGBRegressor(
#             n_estimators=2000,
#             learning_rate=0.03,
#             max_depth=5,
#             subsample=0.8,
#             colsample_bytree=0.8,
#             reg_alpha=0.1,
#             reg_lambda=1.0,
#             #histo
#             tree_method='hist',
#             random_state=42
#             )
#         )

In [ ]:
# model init and search
xgb = XGBRegressor(
    random_state=42,
    tree_method='hist',
    n_jobs=1
)

param_grid = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    verbose=2,
    n_jobs=-1
)


/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 148,991 rows and 26 columns from gridzero-489711.merged_set.test_merge_2017_onward_raw


(XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...),
         temperature_2m_c  wind_speed_100m_ms  wind_gusts_10m_ms  \
 27839                4.8                25.7               21.2   
 141340              14.1                48.9               81.4   
 5474                 6.5         

In [52]:
# loading data
df = load_from_bigquery('gridzero-489711', 'merged_set', 'test_merge_2017_onward_raw')
# df = xgb_train_preproc(df)

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 148,991 rows and 26 columns from gridzero-489711.merged_set.test_merge_2017_onward_raw


In [53]:
# preproc
# defining X and target
# keeping dattime to help plotting
# splitting temporally bc multiple years

df = xgb_train_preproc(df)
target_col = 'carbon_intensity_gCO2_kWh'

# sort by datetime and reset index ooooo
df = df.sort_values('datetime').reset_index(drop=True)

X = df.drop(columns=[target_col, 'datetime'], errors='ignore')
y = df[target_col]

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

X_train = train_df.drop(columns=[target_col, 'datetime'], errors='ignore')
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col, 'datetime'], errors='ignore')
y_test = test_df[target_col]

# keep only num cola to make xgboost happy
feature_cols = X_train.select_dtypes(include='number').columns.tolist()

X_train = X_train[feature_cols]
X_test = X_test[feature_cols]

In [13]:
# grid search: fit and best and pred

grid.fit(X_train, y_train)
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, min_child_weight=1, n_estimators=100, subsample=1.0; total time=   0.6s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, min_child_weight=1, n_estimators=100, subsample=1.0; total time=   0.6s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, min_child_weight=1, n_estimators=100, subsample=0.8; total time=   0.6s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, min_child_weight=1, n_estimators=100, subsample=1.0; total time=   0.6s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, min_child_weight=1, n_estimators=100, subsample=1.0; total time=   0.6s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, min_child_weight=1, n_estimators=100, subsample=0.8; total time=   0.6s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, min_child_weight=1, n_estimators=100, subsample=0.8; tot

In [ ]:
# inspect
best_model = grid.best_estimator_
print(best_model)

best_params = pd.DataFrame(list(grid.best_params_.items()),columns=['parameter', 'value'])
print(best_params)
# best params:

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=5, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=1, num_parallel_tree=None, ...)
          parameter  value
0  colsample_bytree    0.8
1     learning_rate    0.1
2         max_depth    3.0
3  min_child_weight    5.0
4      n_estimators  500.0
5         subsample    1.0


In [ ]:
# improved grid for grid search


# pipeline = pipeline = make_pipeline(
#         StandardScaler(),
#         XGBRegressor(
#             n_estimators=2000,
#             learning_rate=0.03,
#             max_depth=5,
#             subsample=0.8,
#             colsample_bytree=0.8,
#             reg_alpha=0.1,
#             reg_lambda=1.0,
#             #histo
#             tree_method='hist',
#             random_state=42
#             )
#         )

In [54]:
# bigboy params_grid

param_grid_bb = {
    'n_estimators': np.arange(200, 2200, 200),
    'learning_rate': np.arange(0.01, 0.11, 0.01),
    'max_depth': np.arange(3, 11, 1),
    'min_child_weight': np.arange(1, 11, 1),
    'subsample': np.arange(0.6, 1.01, 0.1),
    'colsample_bytree': np.arange(0.6, 1.01, 0.1),
    'gamma': np.arange(0, 1.1, 0.1),
    'reg_alpha': [0, 0.01, 0.05, 0.1, 0.5, 1.0],
    'reg_lambda': [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
}


scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2"
}

In [56]:
# grissearch cross val

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid_bb,
    n_iter=100,
    scoring= scoring,
    refit = 'MAE',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END colsample_bytree=0.9999999999999999, gamma=0.1, learning_rate=0.07, max_depth=4, min_child_weight=1, n_estimators=1000, reg_alpha=0.5, reg_lambda=10.0, subsample=0.7999999999999999; total time=   4.5s
[CV] END colsample_bytree=0.9999999999999999, gamma=0.1, learning_rate=0.07, max_depth=4, min_child_weight=1, n_estimators=1000, reg_alpha=0.5, reg_lambda=10.0, subsample=0.7999999999999999; total time=   4.4s
[CV] END colsample_bytree=0.9999999999999999, gamma=0.1, learning_rate=0.07, max_depth=4, min_child_weight=1, n_estimators=1000, reg_alpha=0.5, reg_lambda=10.0, subsample=0.7999999999999999; total time=   4.5s
[CV] END colsample_bytree=0.9999999999999999, gamma=0.1, learning_rate=0.07, max_depth=4, min_child_weight=1, n_estimators=1000, reg_alpha=0.5, reg_lambda=10.0, subsample=0.7999999999999999; total time=   4.5s
[CV] END colsample_bytree=0.9999999999999999, gamma=0.1, learning_rate=0.07, max_depth=4, min_chi

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END colsample_bytree=0.9999999999999999, gamma=0.9, learning_rate=0.03, max_depth=10, min_child_weight=2, n_estimators=1800, reg_alpha=0.5, reg_lambda=5.0, subsample=0.8999999999999999; total time=  57.3s
[CV] END colsample_bytree=0.9999999999999999, gamma=0.9, learning_rate=0.03, max_depth=10, min_child_weight=2, n_estimators=1800, reg_alpha=0.5, reg_lambda=5.0, subsample=0.8999999999999999; total time=  57.5s
[CV] END colsample_bytree=0.9999999999999999, gamma=0.9, learning_rate=0.03, max_depth=10, min_child_weight=2, n_estimators=1800, reg_alpha=0.5, reg_lambda=5.0, subsample=0.8999999999999999; total time=  58.2s
[CV] END colsample_bytree=0.6, gamma=0.9, learning_rate=0.07, max_depth=10, min_child_weight=2, n_estimators=1800, reg_alpha=0.5, reg_lambda=0.5, subsample=0.7999999999999999; total time=  36.5s
[CV] END colsample_bytree=0.6, gamma=0.9, learning_rate=0.07, max_depth=10, min_child_weight=2, n_estimators=1800, reg_alpha=0.5, reg_lambda=0.5, subsample=0.7999999999999999;

KeyboardInterrupt: 

In [ ]:
# # best model
# best_model = grid.best_estimator_

In [ ]:
# predicting
# y_pred = pipeline.predict(X_test)

In [ ]:
# fit with girdsearch cv
